In [ ]:
import os
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import pandas as pd
pd.set_option('display.max_colwidth', None) # See full text
# Let's verify it worked. This should print "1" and the name of your A6000
print(f"GPUs available to PyTorch: {torch.cuda.device_count()}")
print(f"Current GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 1. Configuration - Maximum VRAM Efficiency
model_id = "Qwen/Qwen2-0.5B-Instruct" 
max_seq_length = 1024
dtype = None # Auto detection
load_in_4bit = True # CHANGED: Now True for 4-bit quantization

# 2. Load Model & Tokenizer via Unsloth in 4-bit
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# 3. Setup LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth", # Crucial for saving VRAM
    random_state=3407,
)

# Load your parquet file
df = pd.read_parquet('data/status_summary_responses.parquet')
dataset = Dataset.from_pandas(df)

def format_to_chat(row):
    return [
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["response"]}
    ]

# 2. Apply to dataframe
# The column itself is named 'messages', which is all Hugging Face needs
df['messages'] = df.apply(format_to_chat, axis=1)
dataset = Dataset.from_pandas(df[['messages']])

# 3. Apply the tokenizer's chat template
def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            msg, # 'msg' is now correctly a list of dicts: [{"role": "user"...}, {"role": "assistant"...}]
            tokenize=False, 
            add_generation_prompt=False
        ) for msg in examples["messages"]
    ]
    return { "text": texts }

# Map the formatting over the dataset
dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# 5. Initialize Unsloth/TRL SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False, 
    args=TrainingArguments(
        output_dir="./qwen-finetuned-0.5B-4bit",
        per_device_train_batch_size=32,
        gradient_accumulation_steps=1,
        learning_rate=2e-4,
        num_train_epochs=1,
        logging_steps=10,
        optim="adamw_8bit", 
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        seed=3407,
    ),
)

# 6. Train
trainer_stats = trainer.train()
print(trainer_stats)

In [ ]:
# 7. Save the Model and Tokenizer
# Note: This saves the LoRA adapters, not the heavily quantized base model weights.
save_path = "models/qwen2-0.5B-4bit-lora-finetuned-status-summary"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
model.save_pretrained_merged("models/qwen2-instruct-0.5B-merged-status-summary", tokenizer, save_method="merged_16bit")
print(f"Model successfully saved to {save_path}")

In [ ]:
import pandas as pd
from IPython.display import display

# 1. Enable 2x faster inference
FastLanguageModel.for_inference(model) 

# 2. Select 10 samples from the dataset to test
test_samples = dataset.select(range(10)) 

results = []

# 1. Define the stopping tokens (Standard EOS + ChatML End Token)
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|im_end|>")
]

for sample in test_samples:
    prompt = sample["messages"][0]["content"]
    ground_truth = sample["messages"][1]["content"]
    
    inference_messages = [
        {"role": "user", "content": prompt}
    ]
    
    inputs = tokenizer.apply_chat_template(
        inference_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    # 2. Pass the terminators to the generation config
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256, 
        use_cache=True,
        temperature=0.7,
        do_sample=True,
        eos_token_id=terminators,          # <--- THE FIX
        pad_token_id=tokenizer.pad_token_id # Good practice to avoid warnings
    )
    
    new_tokens = outputs[0][inputs.shape[-1]:]
    generated_response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    results.append({
        "Original Prompt": prompt[:100] + "...", # Truncated for display
        "Fine-tuned Model Output": generated_response,
        "Ground Truth (Target)": ground_truth
    })

# 3. Display side-by-side in a table
df_results = pd.DataFrame(results)
display(df_results)